JAMfit DIII-D VDE Example, Part 1: Building the Model and the Synthetic Data {#doc_tCurr_jamfit_vde_create}
==========
This example prepares a filament reconstruction of a vertical displacement event (VDE) in DIII-D. It sets up JAMfit on the DIII-D vessel, makes synthetic magnetics data for a plasma current that moves down over 30 ms, builds a reduced wall model, and saves what the reconstruction needs. Part 2 (\ref doc_tCurr_jamfit_vde_reconstruct) reads those files, reconstructs the currents and does the TokaMaker post-processing. It takes a few minutes from a clean folder (about 4 on a laptop); the inductance matrices are then cached, so later runs are faster.

If you are new to JAMfit, start with \ref doc_tCurr_jamfit_one_filament, which shows the same steps on a simple torus.

**What this shows**
1. Placing plasma filaments on a uniform grid inside the DIII-D limiter and adding them to a coil XML.
2. Building a sensor set from the real DIII-D magnetics plus a grid of flux probes (used in part 2 to get the wall's flux on the TokaMaker grid).
3. Setting up JAMfit: the ThinCurr wall model and the sensor and coil mutual inductances.
4. Making synthetic data with the full wall model (`gen_synthetic_data`), with the plasma current given as a moving 2D Gaussian (`setup_fil_timeseries`).
5. Building a reduced wall model from the dominant wall eigenmodes (`create_from_runTD_top_modes`).
6. Saving the plasma current, coil currents and real-sensor signals for part 2.

**The setup**
- **Wall:** `DIIID_meshfile_thincurr.h5`, a ThinCurr mesh of the DIII-D vessel.
- **Coils** (from `DIIID_JAMfit_ex.xml`): the 18 F coils and 6 E coils, with currents held constant in time, then 101 plasma filaments added by this notebook.
- **Sensors:** the 117 real DIII-D magnetics in `floops.loc` (75 magnetic probes and 42 flux loops), then 225 flux probes on a 15 x 15 R-Z grid.
- **TokaMaker mesh:** `DIIID_tokamaker_JAMfit_model.h5`, a 2D cross-section of DIII-D. In this part it gives the limiter contour (for placing the filaments) and the bounding box for the probe grid; JAMfit also loads it for the part 2 post-processing.
- **Plasma current:** a constant 0.49 MA, spread over the filaments as a 2D Gaussian whose centre moves from Z = 0 to Z = -1.1 m.

## Load ThinCurr library

To load the ThinCurr python module we need to tell python where to the module is located. This can be done either through the `PYTHONPATH` environment variable or within a script using `sys.path.append()` as below, where we look for the environement variable `OFT_ROOTPATH` to provide the path to where the OpenFUSIONToolkit is installed (`/Applications/OFT` for binaries on macOS).

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path
import xml.etree.ElementTree as ET
plt.rcParams['figure.figsize'] = (6, 5)
%matplotlib inline

# Use the OFT install pointed to by OFT_ROOTPATH, if set
oft_python_path = os.getenv('OFT_ROOTPATH')
if oft_python_path is not None:
    sys.path.append(os.path.join(oft_python_path, 'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.ThinCurr.JAMfit import JAMfit, get_bbox_grid_pts
from OpenFUSIONToolkit.ThinCurr.sensor import flux_loop, circular_flux_loop
from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh

myOFT = OFT_env(nthreads=4)

## User inputs
All the settings for this example are in the next cell. 

**Note:** The files this notebook writes start with `vde_diiid_` or `DIIID_`, so they do not clash with the one-filament example in the same folder. `floops.loc` (the real DIII-D sensors) is only read, never changed.

In [ ]:
# --- input files ---
meshfile_thincurr = 'DIIID_meshfile_thincurr.h5'          # ThinCurr mesh of the DIII-D vessel
meshfile_tokamaker = 'DIIID_tokamaker_JAMfit_model.h5'    # 2D TokaMaker mesh (limiter, coils)
xml_coils = 'DIIID_JAMfit_ex.xml'                         # ThinCurr coils: the 18 F coils and 6 E coils
real_sensors_file = 'floops.loc'                          # the 117 real DIII-D magnetics

# --- output files ---
xml_name = 'DIIID_JAMfit_wfil_ex.xml'                     # coil XML with the plasma filaments added (read by part 2)
sensors_file = 'vde_diiid_floops_with_probes.loc'         # real sensors, then the flux probes
reduced_filename = 'DIIID_vde_reduced_model_ex.h5'        # reduced wall model (read by part 2)
save_path_dir = 'DIIID_ex_plot_files'                     # ThinCurr history and plot files

B0 = 1.91503608   # vacuum toroidal field at R0 for DIII-D [T]
R0 = 1.69550002   # major radius of DIII-D[m]

# --- filament grid ---
fil_resolution = 10   # number of filament columns across the limiter in R
fil_wall_gap = 0.1    # minimum distance from a filament to the limiter [m]

# --- flux probes ---
probe_n_r = 15        # flux probes along R, on a uniform grid over the TokaMaker mesh's bounding box
probe_n_z = 15        # flux probes along Z (probe_n_r * probe_n_z probes in all); part 2 must use the same probe_n_r, probe_n_z

# --- reduced model ---
num_modes = 4         # wall eigenmodes kept in the reduced model
initial_num_eigs = 35 # wall eigenmodes computed before choosing the num_modes to keep

## Loading the limiter
The filaments must be in the coil XML before the JAMfit object is created, because JAMfit reads that XML. So we first set up a \ref OpenFUSIONToolkit.TokaMaker.TokaMaker "TokaMaker" object on the 2D mesh (read with \ref OpenFUSIONToolkit.TokaMaker.load_gs_mesh "load_gs_mesh()"), only to get the limiter contour `lim_contour`, an (n_pts, 2) array of (R, Z) `[m]`.

In [ ]:
mygs = TokaMaker(myOFT)
mesh_pts, mesh_lc, mesh_reg, coil_dict, cond_dict = load_gs_mesh(meshfile_tokamaker)
mygs.setup_mesh(mesh_pts, mesh_lc, mesh_reg)
mygs.setup_regions(cond_dict=cond_dict, coil_dict=coil_dict)
mygs.setup(order=2, F0=B0*R0)
limiter = mygs.lim_contour   # (n_pts, 2) array of (R, Z) [m]

## Placing the plasma filaments
Here we define following functions `construct_filaments_equal_spacing` and `point_to_segment_dist` to help place filaments within the limiter. These functions are not included within JAMfit. 

The helper `construct_filaments_equal_spacing` below puts points on a uniform R-Z grid, keeps the points inside the limiter, and drops points closer than `tol` to it. The R spacing is $\Delta R = (R_{max} - R_{min}) / (n - 1)$ across the limiter, and the Z spacing is as close to $\Delta R$ as fits a whole number of points between $Z_{min}$ and $Z_{max}$.

In [ ]:
def point_to_segment_dist(px, py, x0, y0, x1, y1):
    '''! Distance from the point (px, py) to the line segment (x0, y0)-(x1, y1).

    @param px float, R of the point `[m]`
    @param py float, Z of the point `[m]`
    @param x0 float, R of the segment start `[m]`
    @param y0 float, Z of the segment start `[m]`
    @param x1 float, R of the segment end `[m]`
    @param y1 float, Z of the segment end `[m]`
    @result float, distance `[m]`
    '''
    dx, dy = x1 - x0, y1 - y0
    if dx == dy == 0:
        # the segment is a single point
        return np.hypot(px - x0, py - y0)
    t = ((px - x0) * dx + (py - y0) * dy) / (dx * dx + dy * dy)
    t = np.clip(t, 0, 1)
    return np.hypot(px - (x0 + t * dx), py - (y0 + t * dy))


def construct_filaments_equal_spacing(limiter, n=10, tol=1E-1, z_buffer=0.0):
    '''! Place filaments on a uniform R-Z grid inside a closed limiter contour.
    The R spacing is `(Rmax - Rmin) / (n - 1)` and the same spacing is used in Z.

    @param limiter numpy.ndarray, limiter contour (R, Z) `[m]`, shape (n_pts, 2)
    @param n int, number of grid points across the limiter in R (default: 10)
    @param tol float, minimum distance from a filament to the limiter `[m]` (default: 0.1)
    @param z_buffer float, shrink the Z range by this much at the top and bottom `[m]` (default: 0.0)
    @result numpy.ndarray, R of the filaments `[m]`, shape (n_fil,)
    @result numpy.ndarray, Z of the filaments `[m]`, shape (n_fil,)

    Example
    -------
    >>> R, Z = construct_filaments_equal_spacing(mygs.lim_contour, n=10, tol=0.1)
    '''
    R_bound, Z_bound = limiter[:, 0], limiter[:, 1]
    Zmin = np.min(Z_bound) + z_buffer
    Zmax = np.max(Z_bound) - z_buffer
    if Zmin >= Zmax:
        raise ValueError('z_buffer={0} is too large, the Z range is empty'.format(z_buffer))
    # R range of the limiter points that lie inside the (shrunk) Z range
    z_mask = (Z_bound >= Zmin) & (Z_bound <= Zmax)
    Rmin, Rmax = np.min(R_bound[z_mask]), np.max(R_bound[z_mask])

    dR = (Rmax - Rmin) / (n - 1)
    nZ = max(2, round((Zmax - Zmin) / dR) + 1)
    RR, ZZ = np.meshgrid(np.linspace(Rmin, Rmax, n), np.linspace(Zmin, Zmax, nZ))
    points = np.column_stack([RR.ravel(), ZZ.ravel()])

    # close the contour if needed, then keep the grid points inside it
    if R_bound[0] != R_bound[-1] or Z_bound[0] != Z_bound[-1]:
        R_bound = np.append(R_bound, R_bound[0])
        Z_bound = np.append(Z_bound, Z_bound[0])
    inside = Path(np.column_stack([R_bound, Z_bound])).contains_points(points)
    points = points[inside]

    # drop points closer than tol to any limiter segment
    if tol > 0.0:
        keep = []
        for r, z in points:
            min_dist = np.min([point_to_segment_dist(r, z, R_bound[i], Z_bound[i], R_bound[i + 1], Z_bound[i + 1])
                               for i in range(len(R_bound) - 1)])
            if min_dist >= tol:
                keep.append([r, z])
        points = np.array(keep)
    return points[:, 0], points[:, 1]

Now placing the filaments and adding each one to the coil XML as its own unnamed `coil_set`. With `fil_resolution = 10` and `fil_wall_gap = 0.1` m this gives 101 filaments. They come after the 24 F and E coils already in the XML, which is the order JAMfit expects (`[non-filament coils | filaments]`); `num_non_fil_coils` is read from the XML. Part 2 finds the filaments again by reading the unnamed coil sets, so do not give them names.

In [ ]:
R, Z = construct_filaments_equal_spacing(limiter, n=fil_resolution, tol=fil_wall_gap)
num_filaments = len(R)

xml_tree = ET.parse(xml_coils)
icoil_tag = xml_tree.getroot().find('./thincurr/icoils')
num_non_fil_coils = len(icoil_tag.findall('coil_set'))   # the shaping coils already in the XML
for r, z in zip(R, Z):
    coil_set = ET.SubElement(icoil_tag, 'coil_set')
    coil = ET.SubElement(coil_set, 'coil')
    coil.set('scale', '1.0')
    coil.text = '{0:.6f}, {1:.6f}'.format(r, z)
ET.indent(xml_tree, space='  ', level=0)
xml_tree.write(xml_name, encoding='utf-8', xml_declaration=True)
print('{0} shaping coils + {1} filaments written to {2}'.format(num_non_fil_coils, num_filaments, xml_name))

fig, ax = plt.subplots()
ax.plot(limiter[:, 0], limiter[:, 1], 'k-')
ax.scatter(R, Z, marker='o', edgecolors='k')
ax.set_xlabel('R [m]')
ax.set_ylabel('Z [m]')
ax.set_aspect('equal', adjustable='box')
ax.set_title('{0} plasma filaments'.format(num_filaments))
plt.show()

## Building sensors
`floops.loc` holds the 117 DIII-D magnetics: 75 Mirnov probes and 42 flux loops. The small reader `read_sensors` below reads them back as \ref OpenFUSIONToolkit.ThinCurr.sensor.flux_loop "flux_loop" objects, from the format written by \ref OpenFUSIONToolkit.ThinCurr.sensor.save_sensors "save_sensors()". 

**Note:** We have provided the DIII-D magnetics as a ready-made sensor file, `floops.loc`, which `setup_JAMfit` could read as it is. We only read it back into sensor objects with `read_sensors` because we want to add the wall flux probes and then write everything to one new file which we then input into `setup_JAMfit`

Part 2 also needs the flux from the wall currents on the TokaMaker grid, to get the total psi for the post-processing. For that we add a nonphysical toroidal flux loop (a "probe", a \ref OpenFUSIONToolkit.ThinCurr.sensor.circular_flux_loop "circular_flux_loop") at each point of a uniform `probe_n_r` x `probe_n_z` grid over the bounding box of the TokaMaker mesh, from \ref OpenFUSIONToolkit.ThinCurr.JAMfit.get_bbox_grid_pts "get_bbox_grid_pts()" (with `verbose=True` it plots the grid). The wall flux varies slowly in space, so a coarse grid is enough (and adding too many of these toroidal flux loops can become computationally taxing): part 2 interpolates it onto every TokaMaker node, calling `get_bbox_grid_pts` again with the same `probe_n_r`, `probe_n_z` to get the probe positions. Each probe adds a sensor to the mutual inductance matrices, so a coarse grid also keeps the setup fast.

**Note:** The real sensors must come first and the probes after them. Part 2 fits only the first 117 signals, and \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.get_wall_psi_tidx "get_wall_psi_tidx()" takes the probes as the sensors after those.

In [ ]:
def read_sensors(filename):
    '''! Read a ThinCurr sensor file (the format written by `save_sensors`) back into flux_loop objects.
    Each sensor is a header line `npts scale name` followed by npts lines of `x y z` `[m]`.

    @param filename str, path to the sensor file
    @result list, flux_loop objects in file order
    '''
    with open(filename, 'r') as fid:
        lines = [line.split() for line in fid if line.strip()]
    num_sensors = int(lines[0][0])
    sensors = []
    i = 1
    for _ in range(num_sensors):
        npts, scale, name = int(lines[i][0]), float(lines[i][1]), lines[i][2]
        pts = np.array(lines[i + 1:i + 1 + npts], dtype=float)
        sensors.append(flux_loop(pts, name, scale=scale))
        i += 1 + npts
    return sensors

In [ ]:
real_sensors = read_sensors(real_sensors_file)
num_real_sensors = len(real_sensors)

probe_pts, _, _, _ = get_bbox_grid_pts(meshfile_tokamaker, myOFT, n_r=probe_n_r, n_z=probe_n_z, verbose=True)
probes = [circular_flux_loop(r, z, 'psi_grid_{0}'.format(i)) for i, (r, z) in enumerate(probe_pts)]
print('{0} real sensors + {1} flux probes'.format(num_real_sensors, len(probes)))

## Set up JAMfit
The \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit "JAMfit" object takes the following inputs:

| Input | Type | Description |
|---|---|---|
| `xml_file` | str | ThinCurr XML configuration file (non-filament coils first, then filaments) |
| `thincurr_meshfile` | str | ThinCurr wall mesh file |
| `tokamaker_meshfile` | str | 2D TokaMaker mesh file of the same device |
| `B0` | float | Vacuum toroidal field at `R0` `[T]` |
| `R0` | float | Reference major radius `[m]` |
| `nthreads` | int | Threads for a new `OFT_env` (needed if `oft_env` is not given) |
| `oft_env` | OFT_env | Existing OFT environment to reuse (optional) |

We pass `oft_env=myOFT`, since only one `OFT_env` may exist per Python process. Creating the object also sets up its own TokaMaker on the 2D mesh, which part 2 uses for the post-processing.

\ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.set_sensors "set_sensors()" writes the sensor list to `sensors_file`. \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.setup_JAMfit "setup_JAMfit()" then builds the ThinCurr model, the sensor and coil mutual inductances, the wall inductance matrix (HODLR) and the wall resistance matrix.

**Note:** This is the slow step. The coil-to-wall inductance and the wall inductance matrix are cached in `vde_diiid_Mcoil.save` and `vde_diiid_L.save`, so a second run is faster. Delete those files if you change the mesh or the coils.

In [ ]:
jam_obj = JAMfit(xml_name, meshfile_thincurr, meshfile_tokamaker, B0=B0, R0=R0, oft_env=myOFT)
sensors_path = jam_obj.set_sensors(real_sensors + probes, floops_path=sensors_file)
jam_obj.setup_JAMfit(sensors_path, plot_files=save_path_dir, mcoil_cache='vde_diiid_Mcoil.save', lmat_cache='vde_diiid_L.save')

## Making synthetic data
We now make pseudo-data like the VDE we want to reconstruct to find the relevant wall mode, which we use to construct a reduced JAMfit object. The F and E coil currents are those of the equilibrium in \ref doc_tMaker_DIIID_ex2.

**Time base and plasma current**
The waveforms are given at 5 time points and ThinCurr interpolates between them. `dt * nsteps` should cover `time_array`: here `dt` = 0.2 ms and `nsteps` = 150, so the run covers 30 ms. The total plasma current stays at 0.49 MA.

In [ ]:
dt = 2.E-4       # time step of the ThinCurr run [s]
nsteps = 150     # number of time steps, so the run covers dt*nsteps = 30 ms
time_array = np.linspace(0, 0.03, 5).reshape(-1, 1)   # waveform time points [s], shape (n_times, 1)
totalip = [493324.5] * len(time_array)                # total plasma current at each time point [A]

**F and E coil currents**
The coil currents `[A]`, in the same order as the coil sets in `DIIID_JAMfit_ex.xml`. They are held constant in time. The `assert` checks that there is one current for each coil in the XML.

In [ ]:
f_coil_currents = {
    'F1A': 1999.5169719827586,
    'F2A': 1031.1705953663798,
    'F3A': -530.3450296336213,
    'F4A': -691.5127963362075,
    'F5A': 12.465672986260786,
    'F6A': -2142.877414772729,
    'F7A': 620.4805397727274,
    'F8A': -975.8820716594835,
    'F9A': 4223.563360873754,
    'F1B': 2213.251346982762,
    'F2B': 1316.7264278017242,
    'F3B': -385.27296605603476,
    'F4B': -2295.533943965518,
    'F5B': 6.891535265692348,
    'F6B': -2792.180113636364,
    'F7B': 754.6947443181815,
    'F8B': -886.0302397629307,
    'F9B': 4667.448286853513}

e_coil_currents = {
    'ECOILA': -16030.96191406249,
    'ECOILB': -15782.150390625005,
    'E567UP': -16030.96191406249,
    'E567DN': -15782.150390625005,
    'E89DN': -15782.150390625005,
    'E89UP': -16030.96191406249}

coil_curr_one_time = list(f_coil_currents.values()) + list(e_coil_currents.values())
assert len(coil_curr_one_time) == num_non_fil_coils, 'coil current list must match the coils in the XML'
coil_curr = np.tile(coil_curr_one_time, (len(time_array), 1))   # shape (n_times, num_non_fil_coils)

**Plasma current on the filaments**
\ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.setup_fil_timeseries "setup_fil_timeseries()" spreads `totalip` over the filaments as a 2D Gaussian centred on $(r_0, z_0)$:

$I_k = I_p \, g_k / \sum_j g_j, \quad g_k = \exp\left( -\frac{(R_k - r_0)^2}{2 \sigma_r^2} - \frac{(Z_k - z_0)^2}{2 \sigma_z^2} \right)$

| Symbol | Meaning |
|---|---|
| $I_k$ | current in filament $k$ `[A]` |
| $I_p$ | total plasma current, `totalip` `[A]` |
| $R_k$, $Z_k$ | position of filament $k$ `[m]` |
| $r_0$, $z_0$ | centre of the current, `r0_list`, `z0_list` `[m]` |
| $\sigma_r$, $\sigma_z$ | Gaussian widths, `sigma_r`, `sigma_z` `[m]` (smaller is more peaked) |

The normalisation makes the filament currents add up to exactly $I_p$ at every time. Moving the centre from Z = 0 down to Z = -1.1 m over the 5 time points gives the VDE. The function returns the current of every coil at every time point, one row per time point, with the columns:

| Columns | Content |
|---|---|
| 0 | time `[s]` |
| 1 to 24 | F and E coil currents `[A]` |
| 25 to 125 | filament currents `[A]` |

You can also build this array yourself, with the same columns, from any waveforms you like.

In [ ]:
num_time_points = len(time_array)
r0_list = 1.7 * np.ones(num_time_points)            # R of the current centre [m]
z0_list = np.linspace(0.0, -1.1, num_time_points)   # Z of the current centre [m], moving down
sigma_r = 2E-1                                      # Gaussian width in R [m]
sigma_z = 2E-1                                      # Gaussian width in Z [m]

final_coil_currs = jam_obj.setup_fil_timeseries(time_array, totalip, coil_curr, r0_list, z0_list, sigma_r, sigma_z, R, Z)
print('current array shape (n_times, 1 + coils + filaments):', final_coil_currs.shape)

**Run with the full wall model**
\ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.gen_synthetic_data "gen_synthetic_data()" runs ThinCurr with the full wall model (the inductance and resistance matrices from `setup_JAMfit`) and records every sensor, probes included. It writes the sensor history to `DIIID_ex_plot_files/floops.hist` and returns it. With `verbose=True` it also builds the XDMF plot files, for viewing in VisIt or ParaView.

In [ ]:
hist_file = jam_obj.gen_synthetic_data(final_coil_currs, dt, nsteps, verbose=True, s_freq=10, p_freq=1)

## Build the reduced wall model
\ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.create_from_runTD_top_modes "create_from_runTD_top_modes()" does three things:
1. It computes `initial_num_eigs` (here 35) eigenmodes of the full wall model.
2. It runs the same waveforms again in a reduced model with all of those modes (the preliminary run, saved as `first_reduced_model_temp.h5`).
3. It keeps the `num_modes` (here 4) modes with the largest weight $w_i = \sum_t \lvert I_i(t) \rvert$, the summed amplitude of mode $i$ over the preliminary run, and saves a reduced model with only those modes to `reduced_filename`.

The weight favours modes that are large and/or long-lived. With `verbose=True` it also plots the mode amplitudes and returns the preliminary run's sensor signals and currents, the basis of the reduced model, the kept mode indices and their weights. Here it keeps modes 3, 12, 14 and 32.

**Note:** You can also build a reduced model from modes you choose yourself with \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.create_reduced_model "create_reduced_model()".

In [ ]:
reduced_obj, sensor_signals, currents, eig_vecs, eig_inds, weights = jam_obj.create_from_runTD_top_modes(
    num_modes, reduced_filename, final_coil_currs, dt, nsteps, initial_num_eigs=initial_num_eigs,
    verbose=True, s_freq=10, p_freq=1)
print('kept wall modes:', eig_inds)

## Save the data for part 2
Part 2 needs, at each time: the total plasma current, the F and E coil currents and the real-sensor signals. They are taken from the preliminary run of `create_from_runTD_top_modes` (the wall with all 35 modes), which records every time step. Only the first `num_real_sensors` signals are saved, since the probes are not real measurements. The time arrays are saved separately for the coils and the sensors, in case they differ.

| File | Content | Shape |
|---|---|---|
| `vde_diiid_totalip.npy` | total plasma current (sum of the filament currents) `[A]` | (151,) |
| `vde_diiid_time_array_coilcurr.npy` | times of the coil currents `[s]` | (151,) |
| `vde_diiid_coil_currs.npy` | F and E coil currents `[A]` | (151, 24) |
| `vde_diiid_time_array_sensors.npy` | times of the sensor signals `[s]` | (151,) |
| `vde_diiid_sensor_measurements.npy` | signals of the 117 real sensors (Mirnov `[T]`, flux loops `[Wb]`) | (151, 117) |

In [ ]:
time_array_coilcurr = currents['time']
coil_currs = currents['coil_curr'][:, :num_non_fil_coils]                   # shaping coils [A]
totalip_to_save = np.sum(currents['coil_curr'][:, num_non_fil_coils:], axis=1)   # sum of filament currents [A]
time_array_sensors = sensor_signals['time']
sensor_measurements = sensor_signals['sensors'][:, :num_real_sensors]

np.save('vde_diiid_totalip.npy', totalip_to_save)
np.save('vde_diiid_time_array_coilcurr.npy', time_array_coilcurr)
np.save('vde_diiid_coil_currs.npy', coil_currs)
np.save('vde_diiid_time_array_sensors.npy', time_array_sensors)
np.save('vde_diiid_sensor_measurements.npy', sensor_measurements)
print('saved {0} time points of {1} sensors and {2} coils'.format(len(time_array_sensors), num_real_sensors, num_non_fil_coils))

## Plotting the plasma current
The filament currents at the first and last saved times (0 and 30 ms): the current centre moves from the midplane down toward the bottom of the vessel.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 5))
for ax, t_idx in zip(axes, [0, -1]):
    sc = ax.scatter(R, Z, c=currents['coil_curr'][t_idx, num_non_fil_coils:] / 1E3, cmap='plasma', edgecolors='k')
    ax.plot(limiter[:, 0], limiter[:, 1], 'k-')
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('R [m]')
    ax.set_title('t = {0:.1f} ms'.format(time_array_coilcurr[t_idx] * 1E3))
    fig.colorbar(sc, ax=ax, label='filament current [kA]')
axes[0].set_ylabel('Z [m]')
plt.tight_layout()
plt.show()

## Next steps
Part 2 (\ref doc_tCurr_jamfit_vde_reconstruct) reconstructs the VDE from the saved signals with the three solvers (least squares, TSVD with a Laplacian penalty, and the direct Laplacian solver), then post-processes the result with TokaMaker (LCFS, q95, li).